In [1]:
!pip install -q transformers>=4.30.0 accelerate>=0.20.3 soundfile librosa jiwer evaluate tensorboard openpyxl peft

In [ ]:
!pip install git+https://github.com/openai/whisper.git

In [ ]:
!pip install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124

In [ ]:
!pip install datasets==3.6.0

In [ ]:
!pip install --upgrade torchao

In [ ]:
!pip uninstall peft torchao -y
!pip install peft==0.11.0

In [7]:
import whisper
import torch
import librosa
import numpy as np
from scipy.io import wavfile
import csv
import json
import os
import pandas as pd
import jiwer
import re
from datasets import load_dataset, DatasetDict, features
from transformers import (
    WhisperFeatureExtractor,
    WhisperTokenizer,
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    GenerationConfig,
    DataCollatorWithPadding,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, TaskType
import evaluate
from dataclasses import dataclass
from typing import Any

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

In [10]:
model_large = whisper.load_model("large").to(device)

100%|██████████████████████████████████████| 2.88G/2.88G [00:24<00:00, 126MiB/s]


In [10]:
train_audio = "/content/drive/MyDrive/object_datasets/train"
validation_audio = "/content/drive/MyDrive/object_datasets/validation"
test_audio = "/content/drive/MyDrive/object_datasets/test"

In [18]:
test_metadata = os.path.join(test_audio, "metadata.csv")
test_metadata_df = pd.read_csv(test_metadata)

In [24]:
def clean(text):
  text = re.sub(r'[^\w\s]', '', text)
  text = text.lower()
  text = re.sub(r'([а-яё])\1{2,}', r'\1', text)
  text = re.sub(r'( )\1{2,}', r'\1', text)
  text = re.sub(r'((ч|д)а(ч|д)а)+', '', text)
  text = re.sub(r'(субтитры субтитры)+', '', text)
  text = text.replace('ё', 'е')
  text = text.strip()
  text = text.replace('субтитры сделал dimatorzok', '')
  text = text.replace('субтитры создавал dimatorzok', '')
  text = text.replace('субтитры создал dimatorzok', '')
  text = text.replace('продолжение следует', '')
  text = text.replace('смотрите продолжение в следующей серии', '')
  text = text.replace('субтитры подогнал симон', '')
  text = text.replace('редактор субтитров асемкин корректор аегорова', '')
  text = text.replace('спасибо за просмотр', '')
  text = text.replace('спасибо за внимание', '')
  text = text.replace('продолжаем', '')
  text = text.replace('добро пожаловать', '')
  text = text.replace('дмитрий шепеллетов', '')
  text = text.replace('подпишись на канал и подписывайтесь на наш канал', '')
  text = text.replace('подпишись', '')
  text = text.replace('добавил субтитры dimatorzok', '')
  text = text.replace('субтитры подогнал игорь негода', '')
  text = text.replace('спасибо за субтитры алексею дубровскому', '')
  text = text.replace('спасибо', '')
  text = text.replace('субтитры делал dimatorzok', '')
  text = text.replace('редактор субтитров асемкин', '')
  text = text.replace('субтитры подогнал dimatorzok', '')
  return text

In [27]:
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [19]:
data = []

for i, row in test_metadata_df.iterrows():
    filename = row["file_name"]

    audio_path = os.path.join(test_audio, filename)

    print(filename)
    result = model_large.transcribe(
        audio_path,
        language="ru",
        task="transcribe",
        fp16=torch.cuda.is_available(),
        temperature=0.0,
        best_of=5,
        beam_size=5,
        patience=2.0,
        compression_ratio_threshold=2.4,
        logprob_threshold=-1.0,
        no_speech_threshold=0.6
    )

    print(result["text"])

    data.append({
        "file_name": filename,
        "transcription": result["text"].strip()
    })

df = pd.DataFrame(data)
# output_path = os.path.join(test_audio, "test_objects_metadata_benchmark.csv")
# df3.to_csv(output_path, index=False, encoding="utf-8")

Object_naming_TMS -18-2-1PictureProperties-4.wav
 Продолжение следует...
Object_naming_TMS -18-2-1PictureProperties-5.wav
 Спасибо.
Object_naming_TMS -18-2-1PictureProperties-6.wav
 Субтитры подогнал «Симон»
Object_naming_TMS -18-2-1PictureProperties-7.wav
 Субтитры сделал DimaTorzok
Object_naming_TMS -18-2-1PictureProperties-8.wav
 Субтитры сделал DimaTorzok
Object_naming_TMS -18-2-1PictureProperties-9.wav
 Субтитры сделал DimaTorzok
Object_naming_TMS -18-2-1PictureProperties-10.wav
 Субтитры сделал DimaTorzok
Object_naming_TMS -18-2-1PictureProperties-11.wav
 Субтитры сделал DimaTorzok
Object_naming_TMS -18-2-1PictureProperties-12.wav
 Субтитры сделал DimaTorzok
Object_naming_TMS -18-2-1PictureProperties-13.wav
 Субтитры сделал DimaTorzok
Object_naming_TMS -18-2-1PictureProperties-14.wav
 Субтитры сделал DimaTorzok
Object_naming_TMS -18-2-1PictureProperties-15.wav
 Субтитры сделал DimaTorzok
Object_naming_TMS -18-2-1PictureProperties-16.wav
 Спасибо за субтитры Алексею Дубровскому!
O

In [21]:
df["correct_transcription"] = test_metadata_df["transcription"]
df

,file_name,transcription,correct_transcription
0,Object_naming_TMS -18-2-1PictureProperties-4.wav,Продолжение следует...,это паутина
1,Object_naming_TMS -18-2-1PictureProperties-5.wav,Спасибо.,это цветок
2,Object_naming_TMS -18-2-1PictureProperties-6.wav,Субтитры подогнал «Симон»,это бутерброд
3,Object_naming_TMS -18-2-1PictureProperties-7.wav,Субтитры сделал DimaTorzok,это вертолет
4,Object_naming_TMS -18-2-1PictureProperties-8.wav,Субтитры сделал DimaTorzok,это кенгуру
...,...,...,...
1162,Object_naming_TMS -34-2-3PictureProperties-29.wav,Субтитры сделал DimaTorzok,это бокал
1163,Object_naming_TMS -34-2-3PictureProperties-30.wav,Продолжение следует...,етъ керру
1164,Object_naming_TMS -34-2-3PictureProperties-31.wav,Субтитры сделал DimaTorzok,это кресло
1165,Object_naming_TMS -34-2-3PictureProperties-32.wav,Редактор субтитров А.Семкин Корректор А.Егорова,это верблюд


In [22]:
df["correct_transcription"] = df["correct_transcription"].fillna('')

In [23]:
predicted = list(df["transcription"])
correct = list(df["correct_transcription"])

print(predicted[:100])
print(correct[:100])

['Продолжение следует...', 'Спасибо.', 'Субтитры подогнал «Симон»', 'Субтитры сделал DimaTorzok', 'Субтитры сделал DimaTorzok', 'Субтитры сделал DimaTorzok', 'Субтитры сделал DimaTorzok', 'Субтитры сделал DimaTorzok', 'Субтитры сделал DimaTorzok', 'Субтитры сделал DimaTorzok', 'Субтитры сделал DimaTorzok', 'Субтитры сделал DimaTorzok', 'Спасибо за субтитры Алексею Дубровскому!', 'Продолжение следует...', 'Продолжение следует...', 'Это красиво.', 'Редактор субтитров А.Семкин Корректор А.Егорова', 'Я тебя шукал.', 'Субтитры сделал DimaTorzok', 'Субтитры сделал DimaTorzok', 'Вот ракетка.', 'Субтитры сделал DimaTorzok', 'Это на ощупь.', 'Это свитер.', 'Это барабан.', 'Субтитры сделал DimaTorzok', 'Субтитры сделал DimaTorzok', 'Субтитры сделал DimaTorzok', 'Кто-то вздрог.', 'Субтитры сделал DimaTorzok', 'Это число.', 'Перекрывайте.', 'Субтитры сделал DimaTorzok', 'Субтитры сделал DimaTorzok', 'Субтитры сделал DimaTorzok', 'Продолжение следует...', 'Субтитры сделал DimaTorzok', 'Этот волнен.

In [25]:
predicted = list(map(clean, predicted))
print(predicted[:100])

['', '', '', '', '', '', '', '', '', '', '', '', '', '', '', 'это красиво', '', 'я тебя шукал', '', '', 'вот ракетка', '', 'это на ощупь', 'это свитер', 'это барабан', '', '', '', 'ктото вздрог', '', 'это число', 'перекрывайте', '', '', '', '', '', 'этот волнен', '', 'это карандаш', '', '', 'это пакет', '', '', '', 'это вот так вот', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', 'это печенье', '', '', '', 'это музыкалка', '', '', '', '', '', 'это продажа', '', '', 'это пикимод', 'это классно', 'это кровать', 'это кепка', 'это куки', '', 'это кучероз', 'это паутина', 'это коляска', 'это качели', 'это акеды', '', 'это лампа', 'это моя', 'это концом', 'это подшум', 'это бинокль', 'это костюм', '', 'это тубусы', 'это жарок', 'это щетка', 'это наручники', '']


In [26]:
correct = list(map(clean, correct))
print(correct[:100])

['это паутина', 'это цветок', 'это бутерброд', 'это вертолет', 'это кенгуру', 'это лампа', 'это колесо', 'это верт ой это светофор', 'это пиджак', 'это банка', 'это стул', 'это чемодан', 'это костер', 'это пистолет', 'это пианино', 'это кроссворд', 'это кувшин', 'это вешалка', 'это кактус', 'это домино', 'это ракетка', 'это щетка', 'это наручники', 'это свитер', 'это барабан', 'это помидор', 'это ключ', 'это пылесос', 'это ведро', 'это жираф', 'это часы', 'это кровать', 'это микрофон', 'это кепка', 'это страус', 'это черепаха', 'это бинокль', 'это клоун', 'это кастюля', 'это карандаш', 'это балалайка', 'это штаны', 'это букет', 'это каска', 'это фен', 'это лягушка', 'это водопад', 'это цыпленок', 'это звезда', 'это корзини', 'это кастрюля', 'это копилка', 'это колбаса', 'это горох', 'это скрепка', 'это аквариум', 'это веркуд', 'это башня', 'это пружина', 'это мешок', 'это коляска', 'это ракета', 'это мотоцикл', 'это качели', 'это маяк', 'это сова', 'это маска', 'это зажигалка', 'это си

In [28]:
benchmark_wer = wer_metric.compute(predictions=predicted, references=correct)
benchmark_cer = cer_metric.compute(predictions=predicted, references=correct)

print(f"benchmark_wer: {benchmark_wer * 100:.1f}%")
print(f"benchmark_cer: {benchmark_cer * 100:.1f}%")

benchmark_wer: 67.6%
benchmark_cer: 63.1%


In [12]:
data3 = []

for i, row in test_metadata_df.iterrows():
    filename = row["file_name"]

    audio_path = os.path.join(test_audio, filename)

    print(filename)
    result = model_large.transcribe(
        audio_path,
        language="ru",
        task="transcribe",
        fp16=torch.cuda.is_available(),
        temperature=0.4,
        best_of=10,
        beam_size=5,
        patience=2.0,
        compression_ratio_threshold=2.4,
        logprob_threshold=-1.0,
        no_speech_threshold=0.6
    )

    print(result["text"])

    data3.append({
        "file_name": filename,
        "transcription": result["text"].strip()
    })

df3 = pd.DataFrame(data3)
output_path = os.path.join(test_audio, "test_objects_metadata_benchmark.csv")
df3.to_csv(output_path, index=False, encoding="utf-8")

Object_naming_TMS -18-2-1PictureProperties-4.wav
 Это паутина.
Object_naming_TMS -18-2-1PictureProperties-5.wav
 Спасибо.
Object_naming_TMS -18-2-1PictureProperties-6.wav
 Это вы заглядываете?
Object_naming_TMS -18-2-1PictureProperties-7.wav
 Субтитры сделал DimaTorzok
Object_naming_TMS -18-2-1PictureProperties-8.wav
 Субтитры сделал DimaTorzok
Object_naming_TMS -18-2-1PictureProperties-9.wav
 Субтитры сделал DimaTorzok
Object_naming_TMS -18-2-1PictureProperties-10.wav
 Субтитры сделал DimaTorzok
Object_naming_TMS -18-2-1PictureProperties-11.wav
 Субтитры сделал DimaTorzok
Object_naming_TMS -18-2-1PictureProperties-12.wav
 Субтитры сделал DimaTorzok
Object_naming_TMS -18-2-1PictureProperties-13.wav
 Это банка.
Object_naming_TMS -18-2-1PictureProperties-14.wav
 Спасибо.
Object_naming_TMS -18-2-1PictureProperties-15.wav
 Субтитры сделал DimaTorzok
Object_naming_TMS -18-2-1PictureProperties-16.wav
 Спасибо за субтитры Алексею Дубровскому!
Object_naming_TMS -18-2-1PictureProperties-17.wav


In [13]:
df3

,file_name,transcription
0,Object_naming_TMS -18-2-1PictureProperties-4.wav,Это паутина.
1,Object_naming_TMS -18-2-1PictureProperties-5.wav,Спасибо.
2,Object_naming_TMS -18-2-1PictureProperties-6.wav,Это вы заглядываете?
3,Object_naming_TMS -18-2-1PictureProperties-7.wav,Субтитры сделал DimaTorzok
4,Object_naming_TMS -18-2-1PictureProperties-8.wav,Субтитры сделал DimaTorzok
...,...,...
1162,Object_naming_TMS -34-2-3PictureProperties-29.wav,Субтитры сделал DimaTorzok
1163,Object_naming_TMS -34-2-3PictureProperties-30.wav,Продолжение следует...
1164,Object_naming_TMS -34-2-3PictureProperties-31.wav,Субтитры сделал DimaTorzok
1165,Object_naming_TMS -34-2-3PictureProperties-32.wav,Продолжение следует...


In [16]:
df3["correct_transcription"] = test_metadata_df["transcription"]
df3

,file_name,transcription,correct_transcription
0,Object_naming_TMS -18-2-1PictureProperties-4.wav,Это паутина.,это паутина
1,Object_naming_TMS -18-2-1PictureProperties-5.wav,Спасибо.,это цветок
2,Object_naming_TMS -18-2-1PictureProperties-6.wav,Это вы заглядываете?,это бутерброд
3,Object_naming_TMS -18-2-1PictureProperties-7.wav,Субтитры сделал DimaTorzok,это вертолет
4,Object_naming_TMS -18-2-1PictureProperties-8.wav,Субтитры сделал DimaTorzok,это кенгуру
...,...,...,...
1162,Object_naming_TMS -34-2-3PictureProperties-29.wav,Субтитры сделал DimaTorzok,это бокал
1163,Object_naming_TMS -34-2-3PictureProperties-30.wav,Продолжение следует...,етъ керру
1164,Object_naming_TMS -34-2-3PictureProperties-31.wav,Субтитры сделал DimaTorzok,это кресло
1165,Object_naming_TMS -34-2-3PictureProperties-32.wav,Продолжение следует...,это верблюд


In [34]:
df3["correct_transcription"] = df3["correct_transcription"].fillna('')

In [35]:
predicted = list(df3["transcription"])
correct = list(df3["correct_transcription"])

print(predicted[:100])
print(correct[:100])

['Это паутина.', 'Спасибо.', 'Это вы заглядываете?', 'Субтитры сделал DimaTorzok', 'Субтитры сделал DimaTorzok', 'Субтитры сделал DimaTorzok', 'Субтитры сделал DimaTorzok', 'Субтитры сделал DimaTorzok', 'Субтитры сделал DimaTorzok', 'Это банка.', 'Спасибо.', 'Субтитры сделал DimaTorzok', 'Спасибо за субтитры Алексею Дубровскому!', 'Это пистолет.', 'Продолжение следует...', 'Это красиво.', 'Редактор субтитров А.Семкин Корректор А.Егорова', 'Я тебя шокую.', 'Продолжение следует...', 'Субтитры сделал DimaTorzok', 'Вот ракетка.', 'Субтитры сделал DimaTorzok', 'Это на ощупь.', 'Это свитер.', 'Это парабан.', 'Спасибо за субтитры Алексею Дубровскому!', 'Это ключ.', 'Это пылесос.', 'Это метро.', 'Субтитры сделал DimaTorzok', 'Это число.', 'Субтитры подогнал «Симон»', 'Спасибо за субтитры Алексею Дубровскому!', 'Субтитры сделал DimaTorzok', 'Субтитры сделал DimaTorzok', 'Продолжение следует...', 'Редактор субтитров А.Семкин Корректор А.Егорова', 'Этот вон он.', 'Спасибо за субтитры Алексею Дубр

In [37]:
predicted = list(map(clean, predicted))
print(predicted[:100])

['это паутина', '', 'это вы заглядываете', '', '', '', '', '', '', 'это банка', '', '', '', 'это пистолет', '', 'это красиво', '', 'я тебя шокую', '', '', 'вот ракетка', '', 'это на ощупь', 'это свитер', 'это парабан', '', 'это ключ', 'это пылесос', 'это метро', '', 'это число', '', '', '', '', '', '', 'этот вон он', '', 'это карандаш', '', '', 'вот это кисть', '', '', '', 'это вот так вот', '', 'это же так', '', '', '', '', '', 'до встречи', '', '', '', '', '', '', '', '', 'это печенье', '', '', '', 'это музыкалка', '', '', '', '', '', 'это продажа', '', '', 'это пикимод', 'это классно', 'это кровать', 'это кепка', 'это куки', 'это то что я', 'это кучерглот', 'это паутина', 'это коляска', 'это качели', 'это акеды', '', 'это лампа', 'это моя', 'это концовка', 'это подшум', 'это бинокль', 'это костюм', '', 'это тубусы', 'это жарок', 'это еще одно', 'это наручники', '']


In [38]:
correct = list(map(clean, correct))
print(correct[:100])

['это паутина', 'это цветок', 'это бутерброд', 'это вертолет', 'это кенгуру', 'это лампа', 'это колесо', 'это верт ой это светофор', 'это пиджак', 'это банка', 'это стул', 'это чемодан', 'это костер', 'это пистолет', 'это пианино', 'это кроссворд', 'это кувшин', 'это вешалка', 'это кактус', 'это домино', 'это ракетка', 'это щетка', 'это наручники', 'это свитер', 'это барабан', 'это помидор', 'это ключ', 'это пылесос', 'это ведро', 'это жираф', 'это часы', 'это кровать', 'это микрофон', 'это кепка', 'это страус', 'это черепаха', 'это бинокль', 'это клоун', 'это кастюля', 'это карандаш', 'это балалайка', 'это штаны', 'это букет', 'это каска', 'это фен', 'это лягушка', 'это водопад', 'это цыпленок', 'это звезда', 'это корзини', 'это кастрюля', 'это копилка', 'это колбаса', 'это горох', 'это скрепка', 'это аквариум', 'это веркуд', 'это башня', 'это пружина', 'это мешок', 'это коляска', 'это ракета', 'это мотоцикл', 'это качели', 'это маяк', 'это сова', 'это маска', 'это зажигалка', 'это си

In [40]:
benchmark_wer = wer_metric.compute(predictions=predicted, references=correct)
benchmark_cer = cer_metric.compute(predictions=predicted, references=correct)

print(f"benchmark_wer: {benchmark_wer * 100:.1f}%")
print(f"benchmark_cer: {benchmark_cer * 100:.1f}%")

benchmark_wer: 63.0%
benchmark_cer: 56.6%


Large модель с temperature=0.4 и best_of=10 имеет более низкие показатели ошибки

То, что CER значительно ниже WER, указывает на то, что модель в принципе слышит верные звуки и звуковые сочетания, но не всегда правильно определяет слово целиком, а значит, дообучение имеет смысл.

Найдём бенчмарки для small модели тоже, так как мы будем её дообучать

In [41]:
model_small = whisper.load_model("small").to(device)

100%|███████████████████████████████████████| 461M/461M [00:04<00:00, 97.4MiB/s]


In [42]:
data4 = []

for i, row in test_metadata_df.iterrows():
    filename = row["file_name"]

    audio_path = os.path.join(test_audio, filename)

    print(filename)
    result = model_small.transcribe(
        audio_path,
        language="ru",
        task="transcribe",
        fp16=torch.cuda.is_available(),
        temperature=0.0,
        best_of=5,
        beam_size=5,
        patience=2.0,
        compression_ratio_threshold=2.4,
        logprob_threshold=-1.0,
        no_speech_threshold=0.6
    )

    print(result["text"])

    data4.append({
        "file_name": filename,
        "transcription": result["text"].strip()
    })

df4 = pd.DataFrame(data4)
# output_path = os.path.join(test_audio, "test_metadata_ru_benchmark_small.csv")
# df4.to_csv(output_path, index=False, encoding="utf-8")

Object_naming_TMS -18-2-1PictureProperties-4.wav
 Что-то подчинало
Object_naming_TMS -18-2-1PictureProperties-5.wav
 Спасибо.
Object_naming_TMS -18-2-1PictureProperties-6.wav
 Папа-папа-папа-папа-папа!
Object_naming_TMS -18-2-1PictureProperties-7.wav
 А-да-да-да-да-да
Object_naming_TMS -18-2-1PictureProperties-8.wav
 Аккуратно!
Object_naming_TMS -18-2-1PictureProperties-9.wav

Object_naming_TMS -18-2-1PictureProperties-10.wav
 Я только не знаю.
Object_naming_TMS -18-2-1PictureProperties-11.wav
 Угу Это же удача
Object_naming_TMS -18-2-1PictureProperties-12.wav
 Офигеть, офигеть!
Object_naming_TMS -18-2-1PictureProperties-13.wav
 Это банка.
Object_naming_TMS -18-2-1PictureProperties-14.wav
 Я не знаю, что это такое. Я не знаю.
Object_naming_TMS -18-2-1PictureProperties-15.wav
 Та-та-та-да
Object_naming_TMS -18-2-1PictureProperties-16.wav
 Это костюм
Object_naming_TMS -18-2-1PictureProperties-17.wav
 Я думаю, что это...
Object_naming_TMS -18-2-1PictureProperties-18.wav
 Удачи!
Object_nam

In [45]:
df4["correct_transcription"] = test_metadata_df["transcription"]

In [46]:
df4

,file_name,transcription,correct_transcription
0,Object_naming_TMS -18-2-1PictureProperties-4.wav,Что-то подчинало,это паутина
1,Object_naming_TMS -18-2-1PictureProperties-5.wav,Спасибо.,это цветок
2,Object_naming_TMS -18-2-1PictureProperties-6.wav,Папа-папа-папа-папа-папа!,это бутерброд
3,Object_naming_TMS -18-2-1PictureProperties-7.wav,А-да-да-да-да-да,это вертолет
4,Object_naming_TMS -18-2-1PictureProperties-8.wav,Аккуратно!,это кенгуру
...,...,...,...
1162,Object_naming_TMS -34-2-3PictureProperties-29.wav,Пока-пока!,это бокал
1163,Object_naming_TMS -34-2-3PictureProperties-30.wav,,етъ керру
1164,Object_naming_TMS -34-2-3PictureProperties-31.wav,Ну что? Что? Что? Что? Что? Что? Что? Что? Что...,это кресло
1165,Object_naming_TMS -34-2-3PictureProperties-32.wav,,это верблюд


In [50]:
df4["correct_transcription"] = df4["correct_transcription"].fillna('')

In [51]:
predicted2 = list(df4["transcription"])
correct2 = list(df4["correct_transcription"])

print(predicted2[:100])
print(correct2[:100])

['Что-то подчинало', 'Спасибо.', 'Папа-папа-папа-папа-папа!', 'А-да-да-да-да-да', 'Аккуратно!', '', 'Я только не знаю.', 'Угу Это же удача', 'Офигеть, офигеть!', 'Это банка.', 'Я не знаю, что это такое. Я не знаю.', 'Та-та-та-да', 'Это костюм', 'Я думаю, что это...', 'Удачи!', 'Присаживайтесь!', 'Продолжение следует...', 'А тебе шоколад', 'Поехали!', 'Аккуратно!', 'Продолжение следует...', 'Подожди.', 'Я помню, что у меня есть шоке.', 'Ходи свитер', 'Эй, супер-бан!', 'Пойдем, пойдем, пойдем.', 'Это конч.', 'Я так обеспечиваюсь', 'Это не другое', 'Поехали. Поехали.', 'Это часы', 'Я закрываю эти.', 'Папа-папа-папа-папа', 'А где петка?', 'Я здесь проживаю', 'А вы чьи походите?', 'Папа-папа-папа-папа', 'Это двои', 'Я только что иду', 'Позвольте подпишись', '', 'Удачи!', 'А, да, да, да, да, да.', 'Спасибо, что...', 'Спасибо.', 'Подожди, подожди.', 'Подпишись на канал и подписывайтесь на наш канал!', 'А что ты делаешь?', 'Это же там.', 'Ого-го-го-го-го!', 'Прекрасно.', 'Это так красиво', 'Па

In [52]:
predicted2 = list(map(clean, predicted2))
print(predicted2[:100])

['чтото подчинало', '', 'папапапапапапапапапа', 'ада', 'аккуратно', '', 'я только не знаю', 'угу это же у', 'офигеть офигеть', 'это банка', 'я не знаю что это такое я не знаю', 'тататада', 'это костюм', 'я думаю что это', 'удачи', 'присаживайтесь', '', 'а тебе шоколад', 'поехали', 'аккуратно', '', 'подожди', 'я помню что у меня есть шоке', 'ходи свитер', 'эй супербан', 'пойдем пойдем пойдем', 'это конч', 'я так обеспечиваюсь', 'это не другое', 'поехали поехали', 'это часы', 'я закрываю эти', 'папапапапапапапа', 'а где петка', 'я здесь проживаю', 'а вы чьи походите', 'папапапапапапапа', 'это двои', 'я только что иду', 'позвольте ', '', 'удачи', 'а да да да да да', ' что', '', 'подожди подожди', '', 'а что ты делаешь', 'это же там', 'огогогогого', 'прекрасно', 'это так красиво', 'папа ты чего папа ты чего', '', '', 'удачи', 'угу', ' барш', '', '', 'вот такая ласка', 'как вы делаете', 'удачи', 'поехали', ' я', 'пошли пошли', '', 'это же колка', 'я не знаю что это', 'я не знаю что это тако

In [53]:
correct2 = list(map(clean, correct2))
print(correct2[:100])

['это паутина', 'это цветок', 'это бутерброд', 'это вертолет', 'это кенгуру', 'это лампа', 'это колесо', 'это верт ой это светофор', 'это пиджак', 'это банка', 'это стул', 'это чемодан', 'это костер', 'это пистолет', 'это пианино', 'это кроссворд', 'это кувшин', 'это вешалка', 'это кактус', 'это домино', 'это ракетка', 'это щетка', 'это наручники', 'это свитер', 'это барабан', 'это помидор', 'это ключ', 'это пылесос', 'это ведро', 'это жираф', 'это часы', 'это кровать', 'это микрофон', 'это кепка', 'это страус', 'это черепаха', 'это бинокль', 'это клоун', 'это кастюля', 'это карандаш', 'это балалайка', 'это штаны', 'это букет', 'это каска', 'это фен', 'это лягушка', 'это водопад', 'это цыпленок', 'это звезда', 'это корзини', 'это кастрюля', 'это копилка', 'это колбаса', 'это горох', 'это скрепка', 'это аквариум', 'это веркуд', 'это башня', 'это пружина', 'это мешок', 'это коляска', 'это ракета', 'это мотоцикл', 'это качели', 'это маяк', 'это сова', 'это маска', 'это зажигалка', 'это си

In [54]:
benchmark_wer2 = wer_metric.compute(predictions=predicted2, references=correct2)
benchmark_cer2 = cer_metric.compute(predictions=predicted2, references=correct2)

print(f"benchmark_wer2: {benchmark_wer2 * 100:.1f}%")
print(f"benchmark_cer2: {benchmark_cer2 * 100:.1f}%")

benchmark_wer2: 109.4%
benchmark_cer2: 89.7%


Приступаем к дообучению

In [ ]:
train_metadata_path = '/content/drive/MyDrive/object_datasets/train/metadata.xlsx'
val_metadata_path = '/content/drive/MyDrive/object_datasets/validation/metadata.xlsx'
test_metadata_path = '/content/drive/MyDrive/object_datasets/test/metadata.xlsx'

df_train = pd.read_excel(train_metadata_path)
df_val = pd.read_excel(val_metadata_path)
df_test = pd.read_excel(test_metadata_path)

df_train.to_csv('/content/drive/MyDrive/object_datasets/train/metadata.csv', index=False, encoding='utf-8')
df_val.to_csv('/content/drive/MyDrive/object_datasets/validation/metadata.csv', index=False, encoding='utf-8')
df_test.to_csv('/content/drive/MyDrive/pbject_datasets/test/metadata.csv', index=False, encoding='utf-8')

In [ ]:
all_datasets = load_dataset(
    "audiofolder",
    data_dir="/content/drive/MyDrive/object_datasets"
)

#здесь для объектов вида AudioDecoder, которые подгрузила функция load_dataset в колонку "audio", подтягивается нормальный аудиофайл,
#переводится в частоту 16кГЦ и сохраняется в память
all_datasets = all_datasets.cast_column("audio", features.Audio(sampling_rate=16000))

print(all_datasets)

Resolving data files:   0%|          | 0/7809 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/688 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1368 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['audio', 'transcription'],
        num_rows: 7055
    })
    validation: Dataset({
        features: ['audio', 'transcription'],
        num_rows: 586
    })
    test: Dataset({
        features: ['audio', 'transcription'],
        num_rows: 1167
    })
})


In [ ]:
all_datasets["train"].features

{'audio': Audio(sampling_rate=16000, mono=True, decode=True, id=None),
 'transcription': Value(dtype='string', id=None)}

По формату теперь всё хорошо

In [ ]:
processor = WhisperProcessor.from_pretrained(
    "openai/whisper-small",
    language="russian",
    task="transcribe"
)

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.48M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/494k [00:00<?, ?B/s]

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

In [ ]:
def prepare_dataset(example):
    audio = example["audio"]

    example["input_features"] = processor(     #передаются спектрограммы аудиозаписей
        audio=audio["array"],
        sampling_rate=audio["sampling_rate"],
        return_tensors="pt"                    #возвращает тензоры PyTorch (что удобнее, чем numpy массивы)
    ).input_features[0]                        #берём первый пример из батча (представляет из себя тензор - цифровой представление спектрограммы)

    if "transcription" not in example or example["transcription"] is None or example["transcription"] == "":

        example["labels"] = []

    else:

      example["labels"] = processor.tokenizer(      #передаются токены эталонных транскрипций
          example["transcription"],
          return_tensors="pt"
      ).input_ids[0]                            #то же самое: берём первый пример из батча, просто у токенизатора другое название поля для этого

    return example

In [ ]:
all_datasets = all_datasets.map(
    prepare_dataset,
    remove_columns=all_datasets["train"].column_names,
    num_proc=1   #последовательная обработка данных, на одном ядре процессора
)

Map:   0%|          | 0/7055 [00:00<?, ? examples/s]

Map:   0%|          | 0/586 [00:00<?, ? examples/s]

Map:   0%|          | 0/1167 [00:00<?, ? examples/s]

In [ ]:
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features):
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(    #на месте паддинга (пустых элементов) ставим значение -100, чтобы они не учитывались в функции потерь
            labels_batch.attention_mask.ne(1), -100
        )

        batch["labels"] = labels

        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

In [ ]:
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
model.generation_config.language = "russian"
model.generation_config.task = "transcribe"

lora_config = LoraConfig(
    r=32,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.87k [00:00<?, ?B/s]

trainable params: 3,538,944 || all params: 245,273,856 || trainable%: 1.4429


In [ ]:
def compute_metrics(pred):

    pred_ids = pred.predictions
    label_ids = pred.label_ids

    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)

    pred_str = [clean(p) for p in pred_str]
    label_str = [clean(l) for l in label_str]

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    cer = cer_metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer, "cer": cer}

In [ ]:
gen_config = GenerationConfig.from_pretrained(      #дублируем ещё раз загрузку с параметрами, чтобы точно не забыл, какой язык нас интересует
    "openai/whisper-small",
    language="russian",
    task="transcribe"
)

training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-small-objects_finetuned",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=1,
    learning_rate=1e-5,
    warmup_steps=50,
    num_train_epochs=5,
    eval_strategy="steps",
    eval_steps=400,
    save_steps=400,
    logging_steps=200,
    load_best_model_at_end=True,
    metric_for_best_model="cer",
    greater_is_better=False,
    save_total_limit=3,
    predict_with_generate=True,
    generation_max_length=225,
    fp16=True,
    report_to=["tensorboard"],
    generation_config=gen_config
)

In [ ]:
trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=all_datasets["train"],
    eval_dataset=all_datasets["validation"],
    data_collator=data_collator,
    processing_class=processor,
    compute_metrics=compute_metrics
)

trainer.train()

Step,Training Loss,Validation Loss,Wer,Cer
400,2.279832,1.600483,0.400000,0.359290
800,1.317106,1.277677,0.337021,0.242813
1200,1.184318,1.147823,0.302979,0.221127
1600,1.049100,1.012211,0.276596,0.170692
2000,0.841837,0.807796,0.268936,0.165927
2400,0.600280,0.543446,0.259574,0.165927
2800,0.368871,0.362117,0.257872,0.163299
3200,0.301644,0.326300,0.255319,0.158206
3600,0.302532,0.316178,0.243404,0.152292
4000,0.293593,0.310000,0.237447,0.149335


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensA

TrainOutput(global_step=4410, training_loss=0.9698357025241635, metrics={'train_runtime': 10557.0956, 'train_samples_per_second': 3.341, 'train_steps_per_second': 0.418, 'total_flos': 1.0359614195712e+19, 'train_loss': 0.9698357025241635, 'epoch': 5.0})

In [ ]:
trainer.state.best_model_checkpoint

'./whisper-small-objects_finetuned/checkpoint-4000'

In [ ]:
trainer.save_model("./whisper-small-objects-finetuned1")
processor.save_pretrained("./whisper-small-objects-finetuned1")

['./whisper-small-objects-finetuned1/processor_config.json']

In [ ]:
test_results = trainer.evaluate(all_datasets["test"])
print(f"Test WER: {test_results['eval_wer'] * 100:.2f}%")
print(f"Test CER: {test_results['eval_cer'] * 100:.2f}%")

Test WER: 30.50%
Test CER: 21.51%


In [ ]:
merged_model = model.merge_and_unload()
merged_model.save_pretrained("/content/drive/MyDrive/whisper-small-objects-final1")
processor.save_pretrained("/content/drive/MyDrive/whisper-small-objects-final1")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

['/content/drive/MyDrive/whisper-small-objects-final1/processor_config.json']

In [ ]:
test_samples = all_datasets["test"].select(range(min(20, len(all_datasets["test"]))))
predictions = trainer.predict(test_samples)

pred_str = processor.batch_decode(predictions.predictions, skip_special_tokens=True)
label_str = processor.batch_decode(predictions.label_ids, skip_special_tokens=True)

pred_norm = [clean(p) for p in pred_str]
label_norm = [clean(l) for l in label_str]

for i in range(len(pred_norm)):
    print(f"Пример {i+1}:")
    print(f"  Эталон: '{label_norm[i]}'")
    print(f"  Предсказание: '{pred_norm[i]}'")
    print()

Пример 1:
  Эталон: 'это паутина'
  Предсказание: 'это паутина'

Пример 2:
  Эталон: 'это цветок'
  Предсказание: 'это цветок'

Пример 3:
  Эталон: 'это бутерброд'
  Предсказание: 'это бисерблот'

Пример 4:
  Эталон: 'это вертолет'
  Предсказание: 'это векторы'

Пример 5:
  Эталон: 'это кенгуру'
  Предсказание: 'это вероятно'

Пример 6:
  Эталон: 'это лампа'
  Предсказание: 'это лампун'

Пример 7:
  Эталон: 'это колесо'
  Предсказание: 'это колеса'

Пример 8:
  Эталон: 'это верт ой это светофор'
  Предсказание: 'это'

Пример 9:
  Эталон: 'это пиджак'
  Предсказание: 'это чел'

Пример 10:
  Эталон: 'это банка'
  Предсказание: 'это банка'

Пример 11:
  Эталон: 'это стул'
  Предсказание: 'это струн'

Пример 12:
  Эталон: 'это чемодан'
  Предсказание: 'это вода'

Пример 13:
  Эталон: 'это костер'
  Предсказание: 'это костер'

Пример 14:
  Эталон: 'это пистолет'
  Предсказание: 'это пистолет'

Пример 15:
  Эталон: 'это пианино'
  Предсказание: 'это камин'

Пример 16:
  Эталон: 'это кроссвор

Тестирование на другом материале (аудио с называнием объектов):

In [11]:
finetuned_model_path = "/content/drive/MyDrive/whisper-small-objects-final1"
finetuned_model = WhisperForConditionalGeneration.from_pretrained(finetuned_model_path)
processor = WhisperProcessor.from_pretrained(finetuned_model_path)

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

In [12]:
def transcribe_audio(audio_path, model, processor, language="ru"):

    audio, sample_rate = librosa.load(audio_path, sr=16000)

    input_features = processor(
        audio,
        sampling_rate=16000,
        return_tensors="pt"
    ).input_features

    input_features = input_features.to(model.device)

    with torch.no_grad():
        predicted_ids = model.generate(
            input_features,
            language=language,
            task="transcribe",
            num_beams=5,
            temperature=0.0,
            max_length=448
        )

    transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

    return transcription

In [13]:
finetuned_model = finetuned_model.to(device)
finetuned_model.eval()

WhisperForConditionalGeneration(
  (model): WhisperModel(
    (encoder): WhisperEncoder(
      (conv1): Conv1d(80, 768, kernel_size=(3,), stride=(1,), padding=(1,))
      (conv2): Conv1d(768, 768, kernel_size=(3,), stride=(2,), padding=(1,))
      (embed_positions): Embedding(1500, 768)
      (layers): ModuleList(
        (0-11): 12 x WhisperEncoderLayer(
          (self_attn): WhisperAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=False)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
          (f

In [16]:
audios = ["/content/Action_naming_TMS-27-2-1PictureProperties-1.wav", "/content/Action_naming_TMS-27-2-1PictureProperties-2.wav",
          "/content/Action_naming_TMS-27-2-1PictureProperties-3.wav", "/content/Action_naming_TMS-27-2-1PictureProperties-4.wav",
          "/content/Action_naming_TMS-27-2-1PictureProperties-5.wav", "/content/Action_naming_TMS-27-2-1PictureProperties-6.wav",
          "/content/Action_naming_TMS-27-2-1PictureProperties-7.wav", "/content/Action_naming_TMS-27-2-1PictureProperties-8.wav",
          "/content/Action_naming_TMS-27-2-1PictureProperties-9.wav", "/content/Action_naming_TMS-27-2-1PictureProperties-10.wav",
          "/content/Action_naming_TMS-27-2-1PictureProperties-11.wav", "/content/Action_naming_TMS-27-2-1PictureProperties-12.wav",
          "/content/Action_naming_TMS-27-2-1PictureProperties-13.wav", "/content/Action_naming_TMS-27-2-1PictureProperties-14.wav",
          "/content/Action_naming_TMS-27-2-1PictureProperties-15.wav", "/content/Action_naming_TMS-27-2-1PictureProperties-16.wav",
          "/content/Action_naming_TMS-27-2-1PictureProperties-17.wav", "/content/Action_naming_TMS-27-2-1PictureProperties-18.wav",
          "/content/Action_naming_TMS-27-2-1PictureProperties-19.wav", "/content/Action_naming_TMS-27-2-1PictureProperties-20.wav",
          "/content/Action_naming_TMS-27-2-1PictureProperties-21.wav", "/content/Action_naming_TMS-27-2-1PictureProperties-22.wav",
          "/content/Action_naming_TMS-27-2-1PictureProperties-23.wav", "/content/Action_naming_TMS-27-2-1PictureProperties-24.wav",
          "/content/Action_naming_TMS-27-2-1PictureProperties-25.wav", "/content/Action_naming_TMS-27-2-1PictureProperties-26.wav",]

In [ ]:
for audio in audios:
  print(transcribe_audio(audio, finetuned_model, processor, language="ru"))

это мочек
это дочка
это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…это…
это судья свистея
это девочка
это брат плачек
это девушка
это мауприсовка
это на троспут
это дядя
это старика
это самолёт
это сторожка
это пожарный тушен
это цеплярка
это коробка
это баранок
это женщина
это пропа пугает
это дночка
это дядя заводик
это рабочий пилет
это...
это девочка
это няня
это лёнашекрасть


In [14]:
model_large = whisper.load_model("large").to(device)

100%|█████████████████████████████████████| 2.88G/2.88G [00:53<00:00, 57.6MiB/s]


In [17]:
for audio in audios:
  try:
    result = model_large.transcribe(
        audio,
        language="ru",
        task="transcribe",
        fp16=torch.cuda.is_available(),
        temperature=0.0,
        best_of=5,
        beam_size=5,
        patience=2.0,
        compression_ratio_threshold=2.4,
        logprob_threshold=-1.0,
        no_speech_threshold=0.6,
        word_timestamps=True
    )

    if result["segments"] and len(result["segments"][0]["words"]) >= 3:
      print(result["segments"][0]["words"][2]["start"] * 1000, result["text"])
    elif result["segments"] and len(result["segments"][0]["words"]) == 2:
      print(result["segments"][0]["words"][1]["start"] * 1000, result["text"])
    elif result["segments"] and len(result["segments"][0]["words"]) == 1:
      print(result["segments"][0]["words"][0]["start"] * 1000, result["text"])
    else:
      print(result["text"])

  except Exception as e:
    print(f"Ошибка с {audio}: {e}")

1500.0  Тут мальчик появился.
3660.0  Субтитры сделал DimaTorzok
1460.0  Субтитры сделал DimaTorzok
1300.0  Пусть судья свистит.
1400.0  Ух, девочка рвёт.
1060.0  Вот брат плачет.
1420.0  Ну, девушка загадает.
1020.0  Вот мама присутствует.
1220.0  Вот матрос погиб.
1780.0  Тут дядя копает.
1140.0  Вот старик.
1220.0  Вот самолет летает.
1260.0  Ну, старушка, давай.
1200.0  И пожарной души.
1860.0  Субтитры создавал DimaTorzok
1100.0  Пусть коровы мочат.
1060.0  Субтитры сделал DimaTorzok
1040.0  Продолжение следует...
1080.0  Продолжение следует...
1140.0  тут дождь как в лад
1280.0  Вот дядя и заводит.
1260.0  Вот рабочий пилит.
1400.0  Субтитры сделал DimaTorzok
1220.0  Вот девочка чья-то.
1280.0  Вот няня купает.
1020.0  Вот ее наше края.


Кажется, что Large модель без дообучения на записях с объектами справялется лучше, чем дообученная на действиях small модель

Большие размеры Whisper (medium, large) дообучить на ту же задачу не получилось из-за нехватки вычислительных ресурсов